## Setup

1. Install Python from the official Python website.

2. Create a virtual environment in a desired path:

   ```bash
   python -m venv .venv
   ```

3. Activate the environment:

   **Windows**
   ```bash
   .venv\Scripts\activate
   ```

   **macOS / Linux**
   ```bash
   source .venv/bin/activate
   ```

4. Install the package from the wheel:

   ```bash
   pip install <path-to-wheel>
   ```

## To get started

Open Python or a Jupyter notebook and run:

```python
from ckan_batch.helpers import get_notebooks, get_excel_template

get_notebooks()        # copies the create/update Jupyter notebooks into the current directory
get_excel_template()   # copies the input Excel template into the current directory
```

- `get_notebooks()` copies the create/update Jupyter notebooks into your current working directory.
- `get_excel_template()` copies the input Excel template into your current working directory.


<h1>Batch Create</h1>

<p>
This notebook explains how to create a batch of records in the PIDINST site.
</p>

<h2>Template file</h2>

<p>
Batch creation uses the PIDINST Excel template. Each row in the template represents one record to be created.
The notebook reads the template, converts each row into a CKAN payload, and then submits the records to the PIDINST site.
</p>



<div style="border:1px solid #f5c2c7; background:#fff3cd; padding:12px 14px; border-radius:6px; margin:12px 0;">
  <h3 style="color:#E1712B"><b>Important notes </b></h3>
  <ol>
  <li>
    For PIDINST, set <b>record_type</b> to <b>"instrument"</b> even when creating platforms.
  Instruments and platforms use the same schema; they are distinguished by the <b>is_platform</b> field.
  </li><br>
  <li>
    The batch create workflow can create both metadata and attachment resources, provided they are correctly defined in the template and supported by the notebook workflow.
  </li><br>
  <li>
</ol>
  
</div>



<h2>Workflow</h2>

<ol>
  <li>
    <b>Step 1. Read Template</b><br>
    Load the Excel template and convert its content into record payloads.
    Any parsing or validation errors found in the template will be reported before submission.
  </li>
  <li>
    <b>Step 2. Create Records</b><br>
    Submit the prepared payloads to the PIDINST site and create the records in CKAN.
  </li>
</ol>

<div style="border:1px solid #b6d4fe; background:#e7f1ff; padding:12px 14px; border-radius:6px; margin:12px 0;">
    <h3 style="color:#1C69A8"><b>Requirements</b></h3>
  <p style="margin-top:0;">
    To run this notebook, you need the following environment variables:
    <b>CKAN_BASE_URL</b> and <b>CKAN_API_KEY</b>.
  </p>

  <p>
    You may set <b>CKAN_BASE_URL</b> directly in the notebook if needed. However, it is recommended to store
    <b>CKAN_API_KEY</b> in a <code>.env</code> file located in the same directory as this notebook.
  </p>

  <p><b>Example <code>.env</code> file:</b></p>

  <pre style="background:#f8f9fa; padding:10px; border-radius:4px; overflow-x:auto; margin:8px 0;"><code>CKAN_BASE_URL=https://pidinst-dev.data.auscope.org.au/
CKAN_API_KEY=&lt;API-KEY from ckan-site&gt;</code></pre>

  <ul style="margin-bottom:0;">
    <li><b>CKAN_BASE_URL</b>: the base URL of the PIDINST website.</li>
    <li><b>CKAN_API_KEY</b>: your API key from the CKAN site. You can create one from your user dashboard.</li>
  </ul>

  <p style="margin-bottom:0; margin-top:10px;">
    Keep your API key secure and rotate it periodically.
  </p>
</div>

In [1]:
import sys
from pprint import pprint
import os
from dotenv import load_dotenv

from ckan_batch.client import CKANClient
from ckan_batch.reader.pidinst import read_pidinst_template

In [2]:
load_dotenv()

CKAN_BASE_URL = os.getenv("CKAN_BASE_URL", "").rstrip("/")
CKAN_API_KEY = os.getenv("CKAN_API_KEY")  # required for Create/Update/Delete (CUD) actions

# INITIALISE CLIENT

In [3]:
ckan_client = CKANClient(CKAN_BASE_URL, apikey=CKAN_API_KEY)

# Step 1. Read Template

In [6]:
res_map = read_pidinst_template(
    excel_path='ignore/test_create.xlsx', 
    client=ckan_client,
    sheet_name="Instruments"
)
pkgs_dict = res_map.records
if len(res_map.errors) > 0:
    print("=============ERRORS==========\n")
    pprint(res_map.errors)

print(len(pkgs_dict))
# pprint(pkgs_dict[0])

1


# Step 2. Create Instruments

**NOTE:** For PIDINST set `record_type` always to "instrument", even for PLATFORMS. Both are based on the same schema, only a boolean `is_platform` separates them

In [7]:
res = ckan_client.create_records(
    records=pkgs_dict,
    make_public=False,
    dry_run = False,
    record_type="instrument"
)

In [8]:
res.failed

[]